# Case 1. Balancing classes
**Applications for Neural Networks in Medicine**<br>
4.11.2024<br>
Sakari Lukkarinen<br>
[Information Technology, Bachelor's Degree](https://www.metropolia.fi/en/academics/bachelors-degrees/information-technology)<br>
[Metropolia University of Applied Sciences](https://www.metropolia.fi/en)

## 1. Introduction

The aim of this Notebook is to demonstrate how you can use either [imbalanced-learn](https://imbalanced-learn.org/stable/index.html), [numpy](https://numpy.org/doc/stable/reference/random/generated/numpy.random.choice.html) or [tensorflow](https://www.tensorflow.org/tutorials/structured_data/imbalanced_data#oversample_the_minority_class) library to deal with classification with imbalanced classes.

## 2. Setup

We need **pandas** to read the dataset and **numpy**, **imblearn**, and **tensorflow** to handle the imbalances in the classes.

To get rid off some tensorflow warnings, we change the [`set_inter_op_parallelism_threads`](https://www.tensorflow.org/api_docs/python/tf/config/threading/set_inter_op_parallelism_threads) to 0 (automatic).

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
tf.config.threading.set_inter_op_parallelism_threads(0)
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN

print(f'tensorflow: {tf.__version__}')

## 3. Dataset
Read in the dataset using `pandas.read_csv` and show the column names.

In [ ]:
datafile = f"/kaggle/input/heart-disease-health-indicators-dataset/heart_disease_health_indicators_BRFSS2015.csv"
df = pd.read_csv(datafile)
df.columns

## 4. Preprocessing

**NOTE:** Before balancing the classes, we need first separate training, validation and test datasets. In this example, we just make the first split between training and test datasets.

In [ ]:
# Split the data into train (80%) and test datasets (20%)
train_df = df.sample(frac=0.8, random_state=0)
test_df = df.drop(train_df.index)

Separate the labels and features and show how many disease and healthy cases are in the training dataset.

In [ ]:
x = train_df.drop(['HeartDiseaseorAttack'], axis = 1)
y = train_df['HeartDiseaseorAttack']

print(f'Disease cases: {sum(y == 1.0):8d}')
print(f'Healthy cases: {sum(y == 0.0):8d}')

### 4.1. Use **imblearn** library
Use the `imblearn.over_sampling.RandomOverSampler` to resample the dataset. Show the number of cases in the resampled labels.

In [ ]:
random_sampler = RandomOverSampler()
x_re, y_re = random_sampler.fit_resample(x, y)

print('Resampled data')
print(f'Disease cases: {sum(y_re == 1.0):8d}')
print(f'Healthy cases: {sum(y_re == 0.0):8d}')

### 4.2. Use **numpy** library
More complex way is to use `numpy.random.choice`.

First we split the dataset to disease and healthy classes.

In [ ]:
x_disease = x[y == 1.0]
x_health = x[y == 0.0]

y_disease = y[y == 1.0]
y_health = y[y == 0.0]

Then we pick up from the disease class equal number of samples as there are in the healthy class.

In [ ]:
# Create indexes
ids = np.arange(len(y_disease))
# Choose randomly as many indexes as there are healthy labels
choices = np.random.choice(ids, len(y_health))

# Resample disease dataset
x_re_disease = x_disease.iloc[choices]
y_re_disease = y_disease.iloc[choices]

# Print the shape of the new datasets
print(f'Resampled disease features: {x_re_disease.shape}')
print(f'Healthy features:           {x_health.shape}')


Lastly concatenate the datasets and shuffle the order.

In [ ]:
# Concatenate features and labels
x_re2 = np.concatenate([x_re_disease, x_health], axis=0)
y_re2 = np.concatenate([y_re_disease, y_health], axis=0)

# Change the order using random.shuffle
order = np.arange(len(y_re2))
np.random.shuffle(order)
x_re2 = x_re2[order]
y_re2 = y_re2[order]

# Show the shape of the resampled features and labels
print(f'Resampled features (all): {x_re2.shape}')
print(f'Resampled labels (all):   {y_re2.shape}')
print(f'Disease cases (all):      {np.sum(y_re2 == 1.0)}')
print(f'Healthy cases (all):      {np.sum(y_re2 == 0.0)}')

### 4.3. Using **tf.data**

> If you're using tf.data the easiest way to produce balanced examples is to start with a positive and a negative dataset, and merge them.
See: [Tensorflow tutorials](https://www.tensorflow.org/tutorials/structured_data/imbalanced_data#oversample_the_minority_class) and [tf.data.guide](https://www.tensorflow.org/guide/data)

We start with splitting the dataset to healthy and disease cases, similarly as with numpy example.

In [ ]:
x_1 = x[y == 1.0]
x_0 = x[y == 0.0]

y_1 = y[y == 1.0]
y_0 = y[y == 0.0]

A helper function to make tensorflow dataset objects from features and labels. BUFFER_SIZE and BATCH_SIZE are parameters for using the dataset.

In [ ]:
BUFFER_SIZE = 100000
BATCH_SIZE = 2048

def make_ds(features, labels):
    ds = tf.data.Dataset.from_tensor_slices((features, labels))
    ds = ds.shuffle(BUFFER_SIZE).repeat()
    return ds

Make separate disease and health datasets and then merge them using `sample_from_datasets`. Notice that in the newest version of tensorflow this method is found from `tf.data.Dataset` as in version 2.6.4 it is still in `tf.data.experimental` sublibrary.

In [ ]:
print(tf.__version__)

In [ ]:
disease_ds = make_ds(x_1, y_1)
health_ds = make_ds(x_0, y_0)

# This works in newer versions of Tensorflow
#resampled_ds = tf.data.Dataset.sample_from_datasets([disease_ds, health_ds], weights=[0.5, 0.5])
# For Tensorflow 2.6.4
resampled_ds = tf.data.experimental.sample_from_datasets([disease_ds, health_ds], weights=[0.5, 0.5])
resampled_ds = resampled_ds.batch(BATCH_SIZE).prefetch(2)

Then we test how the dataset sampler works. We take 3 batches out and check how many disease and healthy cases are in each batch.

In [ ]:
for xb, yb in resampled_ds.take(3):
    print(f'Batch size: {len(yb)}')
    print(f'Disease cases: {np.sum(yb == 1.0)}')
    print(f'Healthy cases: {np.sum(yb == 0.0)}')
    print(f'Proportion: {np.sum(yb == 1.0)/len(yb):.3f}')
    print('')

## Learn more 

- [Over sampling | imbalanced-learn](https://imbalanced-learn.org/stable/over_sampling.html)
- see also [Ch. 2.1.2. From random over-sampling to SMOTE and ADASYN](https://imbalanced-learn.org/stable/over_sampling.html#from-random-over-sampling-to-smote-and-adasyn)
- [Oversampling | Tensorflow tutorials]( (https://www.tensorflow.org/tutorials/structured_data/imbalanced_data#oversample_the_minority_class)